# Volume 4. From Demo To Claim

Question: what has to exist before a notebook observation becomes a claim?

This notebook follows one evidence entry through the repository indexes. The purpose is procedural hygiene: a demo can motivate a question, but claims need scripts, result artifacts, and limitation-aware summaries.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "research" / "claims" / "index.json").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate repository root")
    REPO_ROOT = REPO_ROOT.parent


In [ ]:
claims_index = json.loads((REPO_ROOT / "research" / "claims" / "index.json").read_text())
entry = next(item for item in claims_index["entries"] if item["id"] == "evidence_merge_composition")

pd.DataFrame([{
    "id": entry["id"],
    "status": entry["status"],
    "category": entry["category"],
    "title": entry["title"],
    "primary_artifact": entry["primary_artifact"],
    "notes": entry["notes"],
}])


First check the evidence chain: scripts should exist, result artifacts should exist, and the primary summary should be readable.


In [ ]:
artifact_rows = []
for kind, paths in [
    ("experiment_script", entry.get("experiment_scripts", [])),
    ("result_artifact", entry.get("result_artifacts", [])),
    ("primary_artifact", [entry.get("primary_artifact")]),
]:
    for relative in paths:
        full_path = REPO_ROOT / relative
        artifact_rows.append({
            "kind": kind,
            "path": relative,
            "exists": full_path.exists(),
            "size_bytes": full_path.stat().st_size if full_path.exists() else None,
        })
artifacts = pd.DataFrame(artifact_rows)
artifacts


In [ ]:
coverage = artifacts.groupby(["kind", "exists"]).size().unstack(fill_value=0)
ax = coverage.plot(kind="bar", stacked=True, figsize=(6.8, 3.4), color=["#e15554", "#59a14f"])
ax.set_ylabel("artifact count")
ax.set_title("Evidence-chain artifact coverage")
ax.legend(title="exists", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xticks(rotation=30, ha="right")
plt.show()
plt.close(ax.figure)


Then apply a promotion checklist. Evidence summaries can be strong and still not be formal claims.


In [ ]:
checklist = pd.DataFrame([
    {"gate": "demo exists", "question": "Can a reader reproduce the mechanism visually?", "status": "not sufficient"},
    {"gate": "experiment script", "question": "Is there a maintained script with parameters?", "status": "present"},
    {"gate": "result artifact", "question": "Are outputs archived and linked?", "status": "present"},
    {"gate": "evidence summary", "question": "Is there a written summary of results and scope?", "status": "present"},
    {"gate": "limitations", "question": "Are failure modes and tested regimes explicit?", "status": "needs review"},
    {"gate": "formal claim", "question": "Is the statement promoted with falsification criteria?", "status": "not yet"},
])
checklist


In [ ]:
primary_path = REPO_ROOT / entry["primary_artifact"]
preview = primary_path.read_text(encoding="utf-8").splitlines()[:12]
pd.DataFrame({"line": range(1, len(preview) + 1), "text": preview})


The operational rule: notebooks are allowed to be vivid and fun, but claims should be boringly traceable. When a claim is repeated in docs or a paper note, this evidence chain is the minimum path to inspect.
